# GIVP R - Benchmark Literature Comparison

Comparacao empirica entre **GIVP-full** e **GRASP-only** em 4 funcoes classicas, com 30 rodadas por configuracao.

## 1. Setup

In [ ]:
suppressPackageStartupMessages({
  library(givp)
  library(dplyr)
})

n_runs <- 30L
n_dims <- 10L
max_iters <- 100L

cat(
  sprintf(
    "n_runs=%d, n_dims=%d, max_iters=%d\n",
    n_runs,
    n_dims,
    max_iters
  )
)


## 2. Funcoes benchmark

In [ ]:
sphere <- function(x) sum(x^2)

rosenbrock <- function(x) {
  sum(100 * (x[-1] - x[-length(x)]^2)^2 + (1 - x[-length(x)])^2)
}

rastrigin <- function(x) {
  10 * length(x) + sum(x^2 - 10 * cos(2 * pi * x))
}

ackley <- function(x) {
  n <- length(x)
  -20 * exp(-0.2 * sqrt(sum(x^2) / n)) -
    exp(sum(cos(2 * pi * x)) / n) + 20 + exp(1)
}

benchmarks <- list(
  Sphere = list(
    func = sphere,
    bounds = replicate(n_dims, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Rosenbrock = list(
    func = rosenbrock,
    bounds = replicate(n_dims, c(-5, 10), simplify = FALSE),
    optimum = 0
  ),
  Rastrigin = list(
    func = rastrigin,
    bounds = replicate(n_dims, c(-5.12, 5.12), simplify = FALSE),
    optimum = 0
  ),
  Ackley = list(
    func = ackley,
    bounds = replicate(n_dims, c(-32.768, 32.768), simplify = FALSE),
    optimum = 0
  )
)


## 3. Configuracoes dos algoritmos

In [ ]:
cfg_full <- function(seed, max_iters = 100L) {
  givp::givp_config(
    max_iterations = max_iters,
    alpha = 0.12,
    adaptive_alpha = TRUE,
    alpha_min = 0.08,
    alpha_max = 0.18,
    vnd_iterations = 200L,
    ils_iterations = 10L,
    perturbation_strength = 4L,
    use_elite_pool = TRUE,
    elite_size = 7L,
    path_relink_frequency = 8L,
    use_cache = TRUE,
    cache_size = 10000L,
    early_stop_threshold = 80L,
    use_convergence_monitor = TRUE,
    seed = seed
  )
}

cfg_grasp_only <- function(seed, max_iters = 100L) {
  givp::givp_config(
    max_iterations = max_iters,
    alpha = 0.12,
    adaptive_alpha = FALSE,
    vnd_iterations = 1L,
    ils_iterations = 1L,
    perturbation_strength = 0L,
    use_elite_pool = FALSE,
    use_convergence_monitor = FALSE,
    use_cache = TRUE,
    cache_size = 10000L,
    early_stop_threshold = max_iters,
    seed = seed
  )
}


## 4. Rodar experimento

In [ ]:
records <- list()
idx <- 1L

for (algo in c("GIVP-full", "GRASP-only")) {
  for (fn_name in names(benchmarks)) {
    spec <- benchmarks[[fn_name]]
    for (seed in 0:(n_runs - 1L)) {
      cfg <- if (algo == "GIVP-full") {
        cfg_full(seed, max_iters = max_iters)
      } else {
        cfg_grasp_only(seed, max_iters = max_iters)
      }
      t0 <- Sys.time()
      res <- givp::givp(
        spec$func,
        spec$bounds,
        config = cfg,
        direction = "minimize",
        seed = seed
      )
      elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

      records[[idx]] <- data.frame(
        algorithm = algo,
        function_name = fn_name,
        seed = seed,
        fun = res$fun,
        nit = res$nit,
        nfev = res$nfev,
        time_s = elapsed
      )
      idx <- idx + 1L
    }
  }
}

df <- bind_rows(records)
cat(sprintf("Registros coletados: %d\n", nrow(df)))
head(df)


## 5. Resumo

In [ ]:
summary_df <- df |>
    group_by(function_name, algorithm) |>
    summarise(
        mean_fun = mean(fun),
        sd_fun = sd(fun),
        best_fun = min(fun),
        median_fun = median(fun),
        mean_nfev = mean(nfev),
        mean_time_s = mean(time_s),
        .groups = "drop"
    )

print(summary_df)


## 6. Wilcoxon

In [ ]:
for (fn_name in unique(df$function_name)) {
  a <- df |>
    filter(function_name == fn_name, algorithm == "GIVP-full") |>
    arrange(seed) |>
    pull(fun)
  b <- df |>
    filter(function_name == fn_name, algorithm == "GRASP-only") |>
    arrange(seed) |>
    pull(fun)

  wt <- wilcox.test(a, b, paired = TRUE, alternative = "less", exact = FALSE)
  decision <- ifelse(wt$p.value < 0.05, "SIM", "NAO")
  cat(
    sprintf(
      "%s: W=%.1f p=%.4f %s\n",
      fn_name,
      wt$statistic,
      wt$p.value,
      decision
    )
  )
}


## 7. Exportar JSON

In [ ]:
jsonlite::write_json(
  list(
    metadata = list(n_runs = n_runs, n_dims = n_dims),
    summary = summary_df,
    records = df
  ),
  path = "benchmark_literature_comparison_r_results.json",
  pretty = TRUE,
  auto_unbox = TRUE
)
cat("Arquivo salvo: benchmark_literature_comparison_r_results.json\n")
